In [1]:
import torch
import os
import sys
sys.path.insert(0, "/Users/adrianjuarez/Documents/Covert_lab/Repos/mother_machine_cell_tracker")
import mmtrack.pre_process_mm as pre
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import pandas as pd
from trackastra.model import Trackastra
from trackastra.tracking import graph_to_ctc, graph_to_napari_tracks, write_to_geff
from trackastra.data import example_data_bacteria
import tifffile
import napari
import json
import numpy as np

pyclesperanto_prototype not installed - using CPU (scipy)


/opt/miniconda3/envs/trackastra/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# code needed to help trim tif stacks
def get_tiff_frame_count(file_path):
	"""
	Reads a TIFF file using imread and returns the index of the last time frame (T - 1).
	NOTE: This loads the entire file into memory.
	"""
	try:
		# Load the entire image stack into memory
		img_stack = tifffile.imread(file_path)

		shape = img_stack.shape

		# Assume the time axis (T) is the first dimension
		if len(shape) >= 3:
			# The last index is T - 1
			return shape[0] - 1
		else:
			# Single 2D image
			return 0

	except FileNotFoundError:
		print(f"Error: TIFF file not found at {file_path}")
		return 0
	except Exception as e:
		print(f"Error reading TIFF file {file_path}: {e}")
		return 0
	
def plot_trackastra_kymograph(imgs, ctc_masks, napari_tracks, napari_tracks_graph):
    kymo_imgs = imgs.transpose(1, 0, 2).reshape(382, -1)  # (382, 1800)
    kymo_masks = ctc_masks.transpose(1, 0, 2).reshape(382, -1)

    kymo_tracks = napari_tracks.copy()
    new_x = napari_tracks[:, 1] * 20 + napari_tracks[:, 3]
    kymo_tracks = np.column_stack([
        napari_tracks[:, 0],  # track_id
        np.zeros(len(napari_tracks)),  # dummy time (all in same frame)
        napari_tracks[:, 2],  # y stays the same
        new_x  # new x position
    ])

    v = napari.Viewer()
    v.add_image(kymo_imgs, name='kymograph')
    v.add_labels(kymo_masks, name='masks_kymo')
    v.add_tracks(data=kymo_tracks, graph=napari_tracks_graph, name='tracks_kymo')

def trim_stacks(time_dict, base_path):
    base_path =f'/Users/adrianjuarez/Documents/Covert_lab/Projects/Operon/image_analysis_testing'

    time_range_dict = json.loads(time_dict)
    # print(len(time_range_dict))
    phase_c_str = '0'
    fluor_c_str = '1'

    for folder, fov_dict in time_range_dict.items():    
        for fov_id in fov_dict.keys():
            trench_time_ranges = fov_dict[fov_id]
            print(f"  FOV: {fov_id}, Trenches: {list(trench_time_ranges.keys())}")

            for peak_id, time_info in trench_time_ranges.items():
                base_file_path = os.path.join(base_path, folder, 'hyperstacked', 'drift_corrected', 'rotated',
                                                'mm_channels', 'subtracted')
                path_to_phase_stack = os.path.join(base_file_path,
                                                    f'subtracted_FOV_{fov_id}_region_{peak_id}_c_{phase_c_str}.tif')
                path_to_labeled_stack = os.path.join(base_file_path,
                                                        f'mm3_segmented_subtracted_FOV_{fov_id}_region_{peak_id}_c_{phase_c_str}.tif')
                path_to_fluor_stack = os.path.join(base_file_path,
                                                    f'subtracted_FOV_{fov_id}_region_{peak_id}_c_{fluor_c_str}.tif')
                # --- Dynamic Time Range Assignment ---
                start = time_info['start']
                end = time_info.get('end') 
            
                
                if end is None:
                    # Calculate end frame dynamically using the Phase stack file
                    end = get_tiff_frame_count(path_to_phase_stack)
                    if end == 0:
                        print(f"    WARNING: Could not determine frame count for {peak_id}. Skipping.")
                        continue

                # 2. Read Image Stacks
                try:
                    stack_phase = tifffile.imread(path_to_phase_stack)
                    stack_labeled = tifffile.imread(path_to_labeled_stack)
                    stack_fluor = tifffile.imread(path_to_fluor_stack)
                except FileNotFoundError as e:
                    print(f"    WARNING: Required file not found for {peak_id}: {e}. Skipping.")
                    continue
                # here is where I need to incorporate start and end
                print(start)
                print(end) 
                stack_phase_trimmed = stack_phase[start:end, :, :].copy()
                stack_labeled_trimmed = stack_labeled[start:end, :, :].copy()   
                return stack_phase_trimmed, stack_labeled_trimmed        
                

In [ ]:
# this code loads up all of our datasets, note that it is not needed for track_astra 
path_all_lineages_df = '/Users/adrianjuarez/Documents/Covert_lab/Projects/Operon/tracked_all_cell_data_aggregate_032626.pkl'
path_all_cell_data_df = '/Users/adrianjuarez/Documents/Covert_lab/Projects/Operon/all_cell_data_aggregate_032626.pkl'
all_lineages_df = pd.read_pickle(path_all_lineages_df)
all_cell_data_df = pd.read_pickle(path_all_cell_data_df)

# code to establish some trouble shooting. Experiment_name can be changed to trouble shoot different experiments 
experiment = 'DUMM_gitg068_baeS_100225'
base_path =f'/Users/adrianjuarez/Documents/Covert_lab/Projects/Operon/image_analysis_testing/{experiment}/hyperstacked/drift_corrected/rotated/mm_channels/subtracted'
path_to_phase_stack_dir=f'{base_path}'
path_to_labels_stack_dir =f'{base_path}/mask_kymos'
phase_list = os.listdir(path_to_phase_stack_dir)
mask_list =os.listdir(path_to_labels_stack_dir)




In [3]:
time_dict ='{"DUMM_gitg068_baeS_100225":{"018":{"1185":{"start": 65, "end": 85},"1260":{"start": 10, "end": 30}}},"DUMM_giTG060_064_121425":{"000":{"1343":{"start": 20, "end": 40}}}}'

In [ ]:

base_path =f'/Users/adrianjuarez/Documents/Covert_lab/Projects/Operon/image_analysis_testing'

time_range_dict = json.loads(time_dict)
# print(len(time_range_dict))
phase_c_str = '0'
fluor_c_str = '1'

device = "automatic" 
model = Trackastra.from_pretrained("general_2d_w_SAM2_features", device=device)

for folder, fov_dict in time_range_dict.items():    
    for fov_id in fov_dict.keys():
        trench_time_ranges = fov_dict[fov_id]
        print(f"  FOV: {fov_id}, Trenches: {list(trench_time_ranges.keys())}")

        for peak_id, time_info in trench_time_ranges.items():
            base_file_path = os.path.join(base_path, folder, 'hyperstacked', 'drift_corrected', 'rotated',
                                            'mm_channels', 'subtracted')
            path_to_phase_stack = os.path.join(base_file_path,
                                                f'subtracted_FOV_{fov_id}_region_{peak_id}_c_{phase_c_str}.tif')
            path_to_labeled_stack = os.path.join(base_file_path,
                                                    f'mm3_segmented_subtracted_FOV_{fov_id}_region_{peak_id}_c_{phase_c_str}.tif')
            path_to_fluor_stack = os.path.join(base_file_path,
                                                f'subtracted_FOV_{fov_id}_region_{peak_id}_c_{fluor_c_str}.tif')
            # --- Dynamic Time Range Assignment ---
            start = time_info['start']
            end = time_info.get('end') 
        
            
            if end is None:
                # Calculate end frame dynamically using the Phase stack file
                end = get_tiff_frame_count(path_to_phase_stack)
                if end == 0:
                    print(f"    WARNING: Could not determine frame count for {peak_id}. Skipping.")
                    continue

            # 2. Read Image Stacks
            try:
                stack_phase = tifffile.imread(path_to_phase_stack)
                stack_labeled = tifffile.imread(path_to_labeled_stack)
                stack_fluor = tifffile.imread(path_to_fluor_stack)
            except FileNotFoundError as e:
                print(f"    WARNING: Required file not found for {peak_id}: {e}. Skipping.")
                continue
            # here is where I need to incorporate start and end
            print(peak_id)
            print(start)
            print(end) 
            stack_phase_trimmed = stack_phase[start:end, :, :].copy()
            stack_labeled_trimmed = stack_labeled[start:end, :, :].copy()   
            track_graph, masks_tracked = model.track(stack_phase_trimmed, stack_labeled_trimmed, mode="ilp")
            ctc_tracks, ctc_masks = graph_to_ctc(
            track_graph,masks_tracked,outdir=f'test_040726"')
            
            napari_tracks, napari_tracks_graph, _ = graph_to_napari_tracks(track_graph)

            np.save(f'ctc_masks{fov_id}_{peak_id}.npy', ctc_masks)
            np.save(f'imgs{fov_id}_{peak_id}.npy', stack_phase_trimmed)
            np.save(f'napari_tracks{fov_id}_{peak_id}.npy', napari_tracks)

            with open(f'napari_tracks_graph{fov_id}_{peak_id}.json', "w") as f:
                json.dump(napari_tracks_graph, f)

INFO:trackastra.model.model:Loading model state from /Users/adrianjuarez/Library/Application Support/trackastra/models/general_2d_w_SAM2_features/model.pt
INFO:trackastra.model.model_api:Using device mps
INFO:trackastra.model.model_api:Default batch size = 4 for model on mps.


/Users/adrianjuarez/Library/Application Support/trackastra/models/general_2d_w_SAM2_features already downloaded, skipping.


INFO:trackastra_pretrained_feats.pretrained_features:Using model facebook/sam2.1-hiera-base-plus with mode mean_patches_exact for pretrained feature extraction.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/sam2.1-hiera-base-plus/resolve/main/sam2.1_hiera_base_plus.pt "HTTP/1.1 302 Found"


  FOV: 018, Trenches: ['1185', '1260']
1185
65
85


INFO:root:Loaded checkpoint sucessfully
INFO:trackastra.model.model_api:Predicting weights for candidate graph
INFO:trackastra.data.wrfeat:Extracting features from 20 frames.
INFO:trackastra.model.model_api:Building windows                   
Building windows: 100%|██████████| 17/17 [00:00<00:00, 6225.18it/s]
INFO:trackastra.model.model_api:Predicting windows with batch size 4
Computing associations: 100%|██████████| 5/5 [00:02<00:00,  1.95it/s]
INFO:trackastra.model.model_api:Running greedy tracker
INFO:trackastra.tracking.tracking:Build candidate graph with delta_t=1
INFO:trackastra.tracking.tracking:Added 144 vertices, 302 edges               
INFO:trackastra.tracking.ilp:Using `gt` ILP config.
INFO:motile.solver:Adding NodeSelection cost...
INFO:motile.solver:Adding NodeSelected variables...
INFO:motile.solver:Adding EdgeSelection cost...
INFO:motile.solver:Adding EdgeSelected variables...
INFO:motile.solver:Adding Appear cost...
INFO:motile.solver:Adding NodeAppear variables...
IN


Candidate graph		144 nodes	302 edges
Solution graph		144 nodes	137 edges


100%|██████████| 31/31 [00:00<00:00, 330008.69it/s]
INFO:trackastra_pretrained_feats.pretrained_features:Using model facebook/sam2.1-hiera-base-plus with mode mean_patches_exact for pretrained feature extraction.


1260
10
30


INFO:httpx:HTTP Request: HEAD https://huggingface.co/facebook/sam2.1-hiera-base-plus/resolve/main/sam2.1_hiera_base_plus.pt "HTTP/1.1 302 Found"
INFO:root:Loaded checkpoint sucessfully
INFO:trackastra.model.model_api:Predicting weights for candidate graph
INFO:trackastra.data.wrfeat:Extracting features from 20 frames.


RuntimeError: MPS backend out of memory (MPS allocated: 4.12 GiB, other allocations: 4.35 GiB, max allowed: 9.07 GiB). Tried to allocate 1024.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
stack_phase_trimmed, stack_labeled_trimmed = trim_stacks(time_dict, base_path)

In [ ]:
path_to_mask = f'{base_path}/mm3_segmented_subtracted_FOV_018_region_1185_c_0.tif'
path_to_phase = f'{base_path}/subtracted_FOV_018_region_1185_c_0.tif'
imgs=tifffile.imread(path_to_phase)
masks=tifffile.imread(path_to_mask)

In [ ]:
# this is trackastra used on a specific example 

device = "automatic" 
model = Trackastra.from_pretrained("general_2d_w_SAM2_features", device=device)

# path_to_mask = f'{base_path}/mm3_segmented_subtracted_FOV_018_region_1185_c_0.tif'
# path_to_phase = f'{base_path}/subtracted_FOV_018_region_1185_c_0.tif'
# imgs=tifffile.imread(path_to_phase)
# masks=tifffile.imread(path_to_mask)
imgs, masks = trim_stacks(time_dict, base_path)

track_graph, masks_tracked = model.track(imgs, masks, mode="ilp")
ctc_tracks, ctc_masks = graph_to_ctc(
    track_graph,masks_tracked,outdir=f'018_1185_tracked_ctc"')

# the following code will automatically plot graphs in tif format 
# such that its not in kymograph form, this makes it tricky to see tracks accurately 
# but we can save outputs here

napari_tracks, napari_tracks_graph, _ = graph_to_napari_tracks(track_graph)
# napari_tracks is an ndarray of shape (424, 4)
# napari_tracks_graph is a dict of size = 68 
# tracks_graph is a DiGraph with sizee 424 (424 nodes and 422 edges)

v = napari.Viewer()
v.add_image(imgs)
v.add_labels(ctc_masks)
v.add_tracks(data=napari_tracks, graph=napari_tracks_graph)


#imgs is an ndarraay of shape (90,382,20) that is the original phase images 
# ctc_tracks is a dataframe with shape (70,4) that containes columns label, t1, t2, parent
# ctc_masks is an ndarray that has the same dimensinos as imgs


# this code saves plot-able outputs 
# TO DO need to modify pathing so that it saves where I want things saved
np.save("ctc_masks.npy", ctc_masks)
np.save("imgs.npy", imgs)
np.save("napari_tracks.npy", napari_tracks)

with open("napari_tracks_graph.json", "w") as f:
    json.dump(napari_tracks_graph, f)


In [ ]:
v = napari.Viewer()
v.add_image(stack_phase_trimmed)
v.add_labels(ctc_masks)
v.add_tracks(data=napari_tracks, graph=napari_tracks_graph)


#imgs is an ndarraay of shape (90,382,20) that is the original phase images 
# ctc_tracks is a dataframe with shape (70,4) that containes columns label, t1, t2, parent
# ctc_masks is an ndarray that has the same dimensinos as imgs


# this code saves plot-able outputs 
# TO DO need to modify pathing so that it saves where I want things saved
np.save("ctc_masks.npy", ctc_masks)
np.save("imgs.npy", stack_phase_trimmed)
np.save("napari_tracks.npy", napari_tracks)

with open("napari_tracks_graph.json", "w") as f:
    json.dump(napari_tracks_graph, f)

In [ ]:
# this code takes outputs and reshapes it to kymograph format like we've been plotting 
# will open a napari window 
def plot_trackastra_kymograph(imgs, ctc_masks, napari_tracks, napari_tracks_graph):
    kymo_imgs = imgs.transpose(1, 0, 2).reshape(382, -1)  # (382, 1800)
    kymo_masks = ctc_masks.transpose(1, 0, 2).reshape(382, -1)

    kymo_tracks = napari_tracks.copy()
    new_x = napari_tracks[:, 1] * 20 + napari_tracks[:, 3]
    kymo_tracks = np.column_stack([
        napari_tracks[:, 0],  # track_id
        np.zeros(len(napari_tracks)),  # dummy time (all in same frame)
        napari_tracks[:, 2],  # y stays the same
        new_x  # new x position
    ])

    v = napari.Viewer()
    v.add_image(kymo_imgs, name='kymograph')
    v.add_labels(kymo_masks, name='masks_kymo')
    v.add_tracks(data=kymo_tracks, graph=napari_tracks_graph, name='tracks_kymo')


In [ ]:
napari_tracks, napari_tracks_graph, _ = graph_to_napari_tracks(track_graph)
# napari_tracks is an ndarray of shape (424, 4)
# napari_tracks_graph is a dict of size = 68 
# tracks_graph is a DiGraph with sizee 424 (424 nodes and 422 edges)

v = napari.Viewer()
v.add_image(imgs)
v.add_labels(ctc_masks)
v.add_tracks(data=napari_tracks, graph=napari_tracks_graph)


#imgs is an ndarraay of shape (90,382,20) that is the original phase images 
# ctc_tracks is a dataframe with shape (70,4) that containes columns label, t1, t2, parent
# ctc_masks is an ndarray that has the same dimensinos as imgs

In [ ]:
# all the code that saves 
# need to add a way to save track_graph

np.save("ctc_masks.npy", ctc_masks)
np.save("imgs.npy", imgs)
np.save("napari_tracks.npy", napari_tracks)

with open("napari_tracks_graph.json", "w") as f:
    json.dump(napari_tracks_graph, f)


In [ ]:
# load all data, note that path files may need to be updated
imgs = np.load("imgs.npy")
napari_tracks = np.load("napari_tracks.npy")
ctc_masks = np.load("ctc_masks.npy")


with open("napari_tracks_graph.json") as f:
    napari_tracks_graph = json.load(f)
# (optional) convert keys/values back to int if needed:
napari_tracks_graph = {int(k): int(v) for k, v in napari_tracks_graph.items()}


In [ ]:
# this code takes outputs and reshapes it to kymograph format like we've been plotting 
# will open a napari window 
def plot_trackastra_kymograph(imgs, ctc_masks, napari_tracks, napari_tracks_graph):
    kymo_imgs = imgs.transpose(1, 0, 2).reshape(382, -1)  # (382, 1800)
    kymo_masks = ctc_masks.transpose(1, 0, 2).reshape(382, -1)

    kymo_tracks = napari_tracks.copy()
    new_x = napari_tracks[:, 1] * 20 + napari_tracks[:, 3]
    kymo_tracks = np.column_stack([
        napari_tracks[:, 0],  # track_id
        np.zeros(len(napari_tracks)),  # dummy time (all in same frame)
        napari_tracks[:, 2],  # y stays the same
        new_x  # new x position
    ])

    v = napari.Viewer()
    v.add_image(kymo_imgs, name='kymograph')
    v.add_labels(kymo_masks, name='masks_kymo')
    v.add_tracks(data=kymo_tracks, graph=napari_tracks_graph, name='tracks_kymo')


In [ ]:
plot_trackastra_kymograph(imgs, ctc_masks, napari_tracks, napari_tracks_graph)

In [ ]:
# Get all unique cell tracks
unique_tracks = np.unique(napari_tracks[:, 0])
print(f"Total number of cells tracked: {len(unique_tracks)}")

# For each cell track
for track_id in unique_tracks:
    # Get all timepoints for this cell
    cell_data = napari_tracks[napari_tracks[:, 0] == track_id]
    
    # Extract information
    timepoints = cell_data[:, 1]  # time
    y_coords = cell_data[:, 2]     # y position
    x_coords = cell_data[:, 3]     # x position
    
    print(f"Track {track_id}: {len(timepoints)} timepoints, "
          f"t={timepoints.min():.0f}-{timepoints.max():.0f}")

In [ ]:
def get_lineage_info(napari_tracks_graph):
    """Extract parent-daughter relationships"""
    lineages = []
    
    for daughter_id, parent_id in napari_tracks_graph.items():
        lineages.append({
            'parent': parent_id,
            'daughter': daughter_id
        })
    
    return lineages

# Find all division events
lineages = get_lineage_info(napari_tracks_graph)
print(f"Number of division events: {len(lineages)}")

# Find all daughters of a specific parent
def get_daughters(parent_id, napari_tracks_graph):
    return [d for d, p in napari_tracks_graph.items() if p == parent_id]

# Example: find sisters (cells with same parent)
from collections import defaultdict
parent_to_daughters = defaultdict(list)
for daughter, parent in napari_tracks_graph.items():
    parent_to_daughters[parent].append(daughter)

for parent, daughters in parent_to_daughters.items():
    if len(daughters) > 1:
        print(f"Parent {parent} -> Daughters {daughters}")